# Pre-processing

Module imports

In [1]:
import multiprocessing as mp
import os
import sys
from functools import partial
from itertools import chain

import numpy as np
import pandas as pd
import trimesh
from numba import njit
from scipy.spatial import KDTree
from tqdm import tqdm

# make multiprocessing compatible with macOS
# mp.set_start_method('fork', force=True)

Adding PATH 

In [2]:
# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root added to sys.path


Parsing `.gro` file

In [3]:
from utils import gro_processing as gp

# Data directory can be accessed due to root PATH we set previously
data_file_path = 'data/npt-HK4.gro'
file = os.path.join(project_root, data_file_path)

# Extracts data from .gro file into DataFrame (unsorted)
df_gro, title, num_atoms, box_dimensions = gp.read_gro(file, multiply=10) # convert nm to Å

# Dictionary of molecules {res_id: [(atom_name, np.array([x, y, z])), ...]}
molecules = {}

for res_id in df_gro.index.get_level_values('res_id').unique():
    residue_data = df_gro.xs(res_id, level='res_id')
    
    # Using itertuples() - much faster for large DataFrames
    atom_list = [(row.Index, np.array([row.x, row.y, row.z])) for row in residue_data.itertuples()]
    
    molecules[res_id] = atom_list
    
# molecules # display 

In [5]:
molecules[1]

[('H28', array([16.02,  9.62,  9.12])),
 ('C48', array([15.65,  8.79,  8.52])),
 ('C47', array([16.55,  8.09,  7.73])),
 ('H27', array([17.54,  8.55,  7.56])),
 ('C46', array([16.03,  7.17,  6.83])),
 ('H26', array([16.71,  6.51,  6.26])),
 ('C40', array([14.66,  6.92,  6.63])),
 ('C50', array([13.8 ,  7.77,  7.34])),
 ('H30', array([12.89,  8.09,  6.82])),
 ('C49', array([14.28,  8.64,  8.32])),
 ('H29', array([13.59,  9.11,  9.03])),
 ('N2', array([14.16,  5.78,  5.98])),
 ('C30', array([14.87,  4.92,  5.12])),
 ('C29', array([15.82,  5.19,  4.15])),
 ('H15', array([16.26,  6.19,  4.13])),
 ('C28', array([16.26,  4.27,  3.21])),
 ('C27', array([15.81,  2.93,  3.32])),
 ('H14', array([16.38,  2.11,  2.86])),
 ('C32', array([14.78,  2.67,  4.21])),
 ('H16', array([14.74,  1.65,  4.61])),
 ('C31', array([14.18,  3.65,  4.99])),
 ('C34', array([12.87,  3.82,  5.6 ])),
 ('C33', array([12.99,  5.07,  6.33])),
 ('C35', array([11.83,  5.52,  6.94])),
 ('H17', array([11.92,  6.08,  7.88])),
 

# Setup

Various configs and atom radii data for constructing molecule mesh

In [6]:
# --- CONFIG ---                 
sphere_radius_scale = 2.0                # balls (atoms): 2.0 × van der Waals radius   # idk why 1.5 :(
bond_radius = 0.1                        # sticks (bonds): cylinder radius in Å
sphere_subdiv = 2                        # atom sphere detail (2 is moderate)

# --- Radii (Å) ---
vdw = {"H":1.20,"C":1.70,"N":1.55,"O":1.52,"F":1.47,"P":1.80,"S":1.80,"Cl":1.75,"Na":2.27,"K":2.75,"Ca":2.31}
cov = {"H":0.31,"C":0.76,"N":0.71,"O":0.66,"F":0.57,"P":1.07,"S":1.05,"Cl":1.02,"Na":1.66,"K":2.03,"Ca":1.74}

# --- Element inference ---
hash_atom = {"OW": "O", "HW": "H", "HW1": "H", "HW2": "H"} # Atom name aliases
color_map = {'O': np.array([220, 20, 60, 255], dtype=np.uint8),     # Crimson Red
             'N': np.array([65, 105, 225, 255], dtype=np.uint8),    # Royal Blue
             'C': np.array([128, 128, 128, 255], dtype=np.uint8),   # Gray
             'H': np.array([230, 230, 230, 255], dtype=np.uint8),   # Light Gray
            }

def infer_element(atomname):
    # common water aliases
    if atomname in hash_atom: 
        return hash_atom[atomname]
    
    # simple: first letter, capitalize second if lowercase
    a = ''.join([c for c in atomname if c.isalpha()])   
    
    # join() joins items in an iterable into one string, '' is specified as the separator.
    # isalpha() method returns True if all the characters are alphabet letters (a-z).

    if a == '': 
        return "C"

    if len(a) >= 2 and a[1].islower(): 
        return (a[0]+a[1]).capitalize()
    
    return a[0].upper()

Helper function for MIC related calculations

In [7]:
# --- Minimum Image Convention (vector) ---
@njit(cache=True, fastmath=True)
def mic_vector(dx, box_dimensions):
    """Return minimum-image displacement for vector dx under PBC."""
    k = np.rint(dx / box_dimensions)
    return dx - k * box_dimensions

@njit(cache=True, fastmath=True)
def mic_distance(a, b, box_dimensions):
    """Return MIC distance between two 3D points a,b."""
    return np.linalg.norm(mic_vector(b - a, box_dimensions))

# Helper function to wrap points into the primary box
@njit(cache=True, fastmath=True)
def wrap_points(points, box_dimensions):
    """
    Wrap points into the primary simulation box using periodic boundary conditions.
    """
    return points % box_dimensions


# compile JIT functions
x = y = np.array([1.0, 2.0, 3.0])
_ = mic_vector(x, y) 
_ = mic_distance(x, x, y)
_ = wrap_points(x,y)

# Generating molecule meshes

This part has 3 function with the following uses:
- Building a single molecule mesh
- Looping through all 1501 molecules to draw mesh
- Saving the meshes into `.ply` file as binary

## Ball-and-stick molecule model

Function to build a single molecule mesh

In [8]:
# --- Build ball-and-stick with PBC ---
def build_molecule_ballstick(coords, elements, 
                             vdw, cov, box_length, 
                             sphere_radius_scale=0.3, 
                             sphere_subdiv=2, 
                             bond_radius=0.1):
    """
    Build trimesh ball-and-stick model under periodic boundary conditions.
    """

    # radii arrays
    vdw_r = np.array([vdw.get(e, 1.70) for e in elements])
    cov_r = np.array([cov.get(e, 0.76) for e in elements])
    
    # number of atoms
    n = len(coords)
    
    # --- Process spheres using chunking--- 
    # Base icosphere (unit radius)
    base_sphere = trimesh.creation.icosphere(subdivisions=sphere_subdiv, radius=1.0)
    # Precompute position, radii and number of vertices for each sphere
    coords_wrapped = np.mod(coords, box_length)
    scaled_radii = vdw_r * sphere_radius_scale  
    num_sphere_vertices = len(base_sphere.vertices)
    # Chunk (pre-allocate) array to hold icosphere object
    meshes_sphere = np.empty(n, dtype=object)
    
    for idx, (pos, r) in enumerate(zip(coords_wrapped, scaled_radii)):
        sphere = base_sphere.copy()
        sphere.apply_scale(r)
        sphere.apply_translation(pos)
        sphere_color = color_map.get(elements[idx], color_map.get('C')) # element not found default to grey (C)
        sphere.visual.vertex_colors = np.tile(sphere_color, (num_sphere_vertices, 1))
        meshes_sphere[idx] = sphere 
        
        
    # --- Process cylinder with numba --- 
    # For small molecules roughly < 1500 atoms, brute force is most efficient
    # Base cylinder (for number of faces and color assignment)
    base_cyl = trimesh.creation.cylinder(radius=1.0, height=1.0, sections=24)
    num_bond_faces = len(base_cyl.faces)
    bond_color = color_map.get('H') # light grey
    
    meshes_cylinder = []
    if n < 1500: # brute force
        for i in range(n):
            for j in range(i+1, n):
                d = mic_distance(coords[i], coords[j], box_length)
                thr = 1.2 * (cov_r[i] + cov_r[j])
                if d < thr:
                    # Unwrap j relative to i
                    disp = mic_vector(coords[j] - coords[i], box_length)
                    pos_i = np.mod(coords[i], box_length)
                    pos_j = pos_i + disp  # may fall outside box but correct bond vector
                    seg = np.vstack((pos_i, pos_j))
                    cyl = trimesh.creation.cylinder(radius=bond_radius,
                                                    segment=seg, sections=24)
                    cyl.visual.face_colors = np.tile(bond_color, (num_bond_faces, 1))
                    meshes_cylinder.append(cyl)
    else: # k-d tree 
        tree = KDTree(coords, leafsize=10)
        for i in range(n):
            _nearest_coords, nearest_index = tree.query(coords[i], k=9) # 8 neighbors
            for j in nearest_index[1:]: # Skip itself
                d = mic_distance(coords[i], coords[j], box_length)
                thr = 1.2 * (cov_r[i] + cov_r[j])
                if d < thr:
                    # Unwrap j relative to i
                    disp = mic_vector(coords[j] - coords[i], box_length)
                    pos_i = np.mod(coords[i], box_length)
                    pos_j = pos_i + disp # may fall outside box but correct bond vector
                    seg = np.vstack((pos_i, pos_j))
                    cyl = trimesh.creation.cylinder(radius=bond_radius,
                                                    segment=seg, sections=24)
                    cyl.visual.face_colors = np.tile(bond_color, (num_bond_faces, 1))
                    meshes_cylinder.append(cyl)

    # --- Merge all into one mesh ---
    molecule = trimesh.util.concatenate(meshes_sphere.tolist() + meshes_cylinder)
    return molecule

## Molecules to meshes

Function to parallelize processing all 1501 molecules.

In [9]:
# Helper function process a single molecule 
def process_single_molecule(mol_items,  
                            vdw, cov, box_length,
                            sphere_radius_scale=0.3, 
                            sphere_subdiv=2, 
                            bond_radius=0.1):
    """Process a single molecule and return (mol_id, mesh)"""
    mol_id, atoms = mol_items
    elements = [infer_element(name) for name, _ in atoms]
    coords = np.vstack([pos for _, pos in atoms])
    
    mesh = build_molecule_ballstick(
        coords, elements, vdw, cov, box_length,
        sphere_radius_scale=sphere_radius_scale,
        sphere_subdiv=sphere_subdiv,
        bond_radius=bond_radius
    )
    
    return mol_id, mesh

In [10]:
def molecules_to_meshes_parallel(molecules, 
                                 vdw, cov, box_dimensions,
                                 sphere_radius_scale=0.3,
                                 sphere_subdiv=2,
                                 bond_radius=0.1,
                                 num_processes=None):
    """
    Convert parsed molecules into trimesh meshes.

    Parameters
    ----------
    molecules : dict[int, list[tuple]]
        From parse_gro(): molecules[i] = [(atomname, coords), ...]
        coords must be in Å
    box_dimensions : np.ndarray
        Simulation box_dimensions (Å), shape (3,) for orthorhombic or (3,3) for triclinic
    vdw, cov : dict
        Van der Waals and covalent radii
    sphere_radius_scale : float
        Scaling factor for atom radii
    sphere_subdiv : int
        Subdivisions for icosphere (mesh resolution)
    bond_radius : float
        Cylinder radius for bonds
    num_processes: int/None
        Number of CPU cores to use (all if not specified)

    Returns
    -------
    mol_meshes : dict[int, trimesh.Trimesh]
        Dictionary of molecule meshes keyed by mol_id
    """
    # assume orthorhombic box for now
    if box_dimensions.shape == (3,):
        box_length = box_dimensions
    else:
        raise NotImplementedError("Triclinic box handling not yet implemented")
    
    
    # --- Multiprocessing ---
    # number of processes (use all CPUs if not specified)
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    # partial function with fixed parameters
    process_func = partial(
        process_single_molecule,
        vdw=vdw,
        cov=cov,
        box_length=box_length,
        sphere_radius_scale=sphere_radius_scale,
        sphere_subdiv=sphere_subdiv,
        bond_radius=bond_radius
    )

    # parallel execution
    mol_items = list(molecules.items())
    num_mol = len(mol_items)
    
    with mp.Pool(processes=num_processes) as pool:
        tqdm_iterator = tqdm(
            pool.imap(process_func, mol_items),
            total=num_mol,
            desc=f'Processing {num_mol} molecules with {num_processes} logical cores',
            colour='#7BC8F6'
        )
        
        mol_meshes = dict(tqdm_iterator) # initialise progress bar
        
    return mol_meshes

In [11]:
# Now mol_meshes is {1: Trimesh(...), 2: Trimesh(...), ..., 1501: Trimesh(...)}
# Takes ~ 50.0 seconds
mol_meshes = molecules_to_meshes_parallel(molecules, vdw, cov, box_dimensions, num_processes=None)
print(len(mol_meshes))        # 1501
print(mol_meshes[1])          # trimesh.Trimesh object

Processing 1501 molecules with 8 logical cores: 100%|██████████| 1501/1501 [00:58<00:00, 25.54it/s]

1501
<trimesh.Trimesh(vertices.shape=(18308, 3), faces.shape=(35904, 3))>


## Export molecule meshes to `.npz` file

Testing Single molecule

In [ ]:
# 1. Create and color the icosphere (same as before)
mesh = trimesh.creation.icosphere(subdivisions=2, radius=1.0)

# Assign a bright red color to every face (RGBA, uint8)
num_faces = len(mesh.faces)
red_color = np.array([255, 0, 0, 255], dtype=np.uint8)
face_colors = np.tile(red_color, (num_faces, 1))
mesh.visual.face_colors = face_colors

# 2. Extract the data in NumPy form
# These are the three fundamental arrays you need for the mesh and color.
vertices = mesh.vertices.copy()
faces = mesh.faces.copy()
colors = mesh.visual.face_colors.copy()

# 3. Use the numpy function to export to an NPZ file
file_name = 'raw_icosphere_data.npz'

# numpy.savez_compressed saves multiple arrays into a single zipped file
# We assign a key (like 'vertices') to each array.
np.savez_compressed(
    file_name,
    vertices=vertices,
    faces=faces,
    face_colors=colors
)

print(f"Mesh data successfully exported to {file_name}")
print(f"File size with compression: {os.path.getsize(file_name) / 1024:.2f} KB")

Mesh data successfully exported to raw_icosphere_data.npz
File size with compression: 2.31 KB


In [ ]:
# Load the NumPy arrays from the NPZ file
loaded_data = np.load(file_name)

# Extract the arrays using the keys we defined during save
loaded_vertices = loaded_data['vertices']
loaded_faces = loaded_data['faces']
loaded_colors = loaded_data['face_colors']

print("\nData loaded from NPZ:")
print(f"Vertices shape: {loaded_vertices.shape}")
print(f"Faces shape: {loaded_faces.shape}")
print(f"Colors shape: {loaded_colors.shape}")


# 4. Create the new Trimesh object using the loaded arrays
recreated_mesh = trimesh.Trimesh(
    vertices=loaded_vertices,
    faces=loaded_faces,
    face_colors=loaded_colors,  # Pass the loaded color array
    process=True
)

recreated_mesh.show()

# --- Verification ---

# print("\nRecreated Trimesh Object Verification:")
# print(f"Type: {type(recreated_mesh)}")
# print(f"First face color: {recreated_mesh.visual.face_colors[0]}")


Data loaded from NPZ:
Vertices shape: (162, 3)
Faces shape: (320, 3)
Colors shape: (320, 4)


---

No parallelization

In [ ]:
# File names and such
num_meshes = len(mol_meshes)
data_file_name = os.path.basename(data_file_path).rsplit('.', 1)[0] # e.g., 'npt-HK4'
save_file_name = f'{data_file_name}_meshes.npz'

# This dictionary will hold all the arrays from all 1501 meshes
all_mesh_data = {}

for i, mesh in mol_meshes.items():
    # Extract individual mesh data
    vertices = mesh.vertices.copy()
    faces = mesh.faces.copy()
    colors = mesh.visual.face_colors.copy()

    # --- Key Step: Store the arrays with unique keys ---
    mesh_key_prefix = f'mesh_{i:04d}' # e.g., 'mesh_0000', 'mesh_0001', etc.
    
    all_mesh_data[f'{mesh_key_prefix}_vertices'] = mesh.vertices.copy()
    all_mesh_data[f'{mesh_key_prefix}_faces'] = mesh.faces.copy()
    all_mesh_data[f'{mesh_key_prefix}_colors'] = mesh.visual.face_colors.copy()

print(f"Collected {len(all_mesh_data)} NumPy arrays.")

# The dictionary unpacking (**) passes all key-value pairs as keyword arguments
np.savez_compressed(save_file_name, **all_mesh_data)

print(f"Successfully saved all {num_meshes} meshes to a single file: {save_file_name}")
print(f"File size: {os.path.getsize(save_file_name) / (1024*1024):.2f} MB")

Collected 4503 NumPy arrays.
Successfully saved all 1501 meshes to a single file: npt-HK4_meshes.npz
File size: 464.17 MB


With parallelization

In [ ]:
import threading
import itertools
import time

class Spinner:
    def __init__(self, message="Working..."):
        self._message = message
        self._done = False
        self._thread = threading.Thread(target=self._animate)

    def _animate(self):
        spinner = itertools.cycle(['.', '..', '...'])
        while not self._done:
            # Use '\r' to return to the start of the line, overwriting previous text
            sys.stdout.write(f'\r{self._message}{next(spinner)}   ') 
            sys.stdout.flush() 
            time.sleep(0.5)

        # Final message to show completion and move to a new line
        sys.stdout.write(f'\rDone: {self._message} complete!   \n')
        sys.stdout.flush()

    def start(self):
        """Starts the spinner animation in a separate thread."""
        self._thread.start()

    def stop(self):
        """Stops the spinner and waits for the thread to finish."""
        self._done = True
        self._thread.join()

In [ ]:
def extract_mesh_data(mol_items):
    """
    Extract singular mesh data: mol_id, vertices, faces, and colors.

    Parameters
    ----------
    mol_items : tuple
        A tuple containing the molecule ID (int) and the corresponding
        molecular mesh (trimesh.Trimesh object).

    Returns
    -------
    None
        The function performs a disk write operation and does not return a value.
    """
    mol_id, mesh = mol_items

    # Extract individual mesh data
    vertices = mesh.vertices.copy()
    faces = mesh.faces.copy()
    colors = mesh.visual.face_colors.copy()
    
    # Store the arrays with unique keys
    mesh_key_prefix = f'mesh_{mol_id:04d}'
    data = {
        f'{mesh_key_prefix}_vertices': vertices,
        f'{mesh_key_prefix}_faces': faces,
        f'{mesh_key_prefix}_colors': colors
    }
    return data


def export_meshes(mol_meshes, path, export_format='npz', num_processes=None, context='spawn'):
    """
    Export molecular meshes to a specified file format.

    Parameters
    ----------
    mol_meshes : dict
        A dictionary where keys are molecule IDs (int) and values are
        the corresponding molecular meshes (trimesh.Trimesh objects).
    export_format : str
        The file format used for exporting the meshes (e.g., 'npz').

    Returns
    -------
    None
        The function performs a disk write operation and does not return a value.
    """
    # Generate output file (or directory) name
    name = os.path.basename(path).rsplit('.', 1)[0] # e.g., 'npt-HK4'
    output_file = f'{name}_meshes.npz'
    
    # Converting for multiprocessing
    mol_items = list(mol_meshes.items())
    num_mol = len(mol_meshes)
    
    # --- Multiprocessing ---
    # number of processes (use all CPUs if not specified)
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    all_mesh_data = {}
    ctx = mp.get_context(context)
    with ctx.Pool(processes=num_processes) as pool:
        # Progress bar
        tqdm_iterator = tqdm(
            pool.imap(extract_mesh_data, mol_items),
            total=num_mol,
            desc=f'Extracting mesh data with {num_processes} logical cores',
            colour='#7BC8F6'
        )
        # Iterate over and add results to dictionary
        for data_dict in tqdm_iterator:
            all_mesh_data.update(data_dict)
    print(f'Converted {num_mol} molecular meshes into {len(all_mesh_data)} numpy arrays.')
    
    # Save to .npz file
    loading = Spinner(f"Saving all meshes to {output_file}")
    loading.start()
    np.savez_compressed(output_file, **all_mesh_data)
    loading.stop()
    
    print(f"Successfully saved all meshes to a single file: {output_file}")
    print(f"File size: {os.path.getsize(output_file) / (1024*1024):.2f} MB")

In [ ]:
export_meshes(mol_meshes, data_file_name, export_format='npz', num_processes=None, context='fork')

# Import mol_meshes from `.npz` files

No parallel

In [ ]:
def reconstruct_all_meshes(file_path):
    """
    Extracts all mesh components from loaded_data, reconstructs trimesh objects,
    and stores them in a dictionary using 1-based indexing.
    
    This method assumes that each mesh has 3 separate data types, vertices, faces, and colors.

    Args:
        file_path (str): The path to the NPZ file.

    Returns:
        dict: A dictionary of {mesh_id: trimesh.Trimesh object}.
    """
    
   # Load the data inside the function
    try:
        loaded_data = np.load(file_path)
    except FileNotFoundError:
        print(f"Error: File not found at path: {file_path}")
        return {}

    # Calculate the number of meshes automatically
    total_arrays = len(loaded_data.files)
    # Integer division by 3 gives the total number of unique meshes
    num_meshes = total_arrays // 3 
    
    # This will hold the result: {1: mesh_0, 2: mesh_1, ...}
    meshes_dict = {}

    # Iterate through all mesh indices (1 to NUM_MESHES)
    for i in range(1, num_meshes + 1):
        
        # 1. Generate the unique key prefix using 4-digit formatting
        key_prefix = f'mesh_{i:04d}'
        
        # 2. Extract the three required arrays using the generated prefix
        try:
            loaded_vertices = loaded_data[f'{key_prefix}_vertices']
            loaded_faces = loaded_data[f'{key_prefix}_faces']
            loaded_colors = loaded_data[f'{key_prefix}_colors']
        except KeyError as e:
            print(f"Error: Missing key {e} for mesh index {i}. Aborting reconstruction.")
            return meshes_dict # Return what was processed so far
            
        # 3. Reconstruct the trimesh object
        reconstructed_mesh = trimesh.Trimesh(
            vertices=loaded_vertices,
            faces=loaded_faces,
            face_colors=loaded_colors # Note: face_colors is used for coloring faces
        )
        
        # 4. Store the mesh in the dictionary using a 1-based ID
        mesh_id = i
        meshes_dict[mesh_id] = reconstructed_mesh

    return meshes_dict

# Run the function to get the dictionary of meshes
all_reconstructed_meshes = reconstruct_all_meshes('npt-HK4_meshes.npz')

In [ ]:
all_reconstructed_meshes[12].show()

---

### Parallel

Overhead to create the global shared variable is much longer than extraction time

In [ ]:
import numpy as np
import trimesh
import multiprocessing as mp
from functools import partial
from tqdm import tqdm

def init_worker(file_path):
    """
    INITIALIZER FUNCTION: Called once when a worker process starts.
    This loads the NPZ file object in that process's memory space.
    """
    global _LOADED_DATA
    try: # np.load() is called ONCE per worker. This loads the header (lazy).
        _LOADED_DATA = np.load(file_path)
    except Exception as e: # Handle load failure gracefully
        print(f"Worker initialization failed for {file_path}: {e}", file=sys.stderr)
        _LOADED_DATA = None

def import_one_mesh(mol_id):
    """
    Extracts mesh components for a single molecule from loaded .npz data,
    reconstructs the trimesh object, and returns it.

    Parameters
    ----------
        mol_id : int
            The ID of the molecule to extract (1-based indexing).
        loaded_data : np.lib.npyio.NpzFile 
            The loaded NPZ data.

    Returns
    -------
        reconstructed_mesh : trimesh.Trimesh
            The reconstructed trimesh object for the specified molecule.
    """
    global _LOADED_DATA
    if _LOADED_DATA is None:
        # Cannot proceed if initialization failed
        return None
    
    key_prefix = f'mesh_{mol_id:04d}'
    try: # Extract the three required data vertices, faces, colors
        loaded_vertices = _LOADED_DATA[f'{key_prefix}_vertices']
        loaded_faces = _LOADED_DATA[f'{key_prefix}_faces']
        loaded_colors = _LOADED_DATA[f'{key_prefix}_colors']
    except KeyError as e:
        print(f"Error: Missing {key_prefix}. Skipping this molecule.")
        return None

    # Reconstruct the trimesh object
    reconstructed_mesh = trimesh.Trimesh(
        vertices=loaded_vertices,
        faces=loaded_faces,
        face_colors=loaded_colors  # Note: face_colors is used for coloring faces
    )
    return {mol_id: reconstructed_mesh}


def import_meshes(file, num_processes=None, context='spawn'):
    """
    Extracts all mesh components from .npz, then reconstructs trimesh objects,
    and stores them in a dictionary using 1-based indexing.
    
    This method assumes that each mesh has 3 separate data types, vertices, faces, and colors.

    Parameters
    ----------
        file_path (str): The path to the NPZ file. (e.g. 'npt-HK4_meshes.npz')

    Returns
    -------
        dict: A dictionary of {mesh_id: trimesh.Trimesh object}.
    """
   # Note: If load a .npz file, it becomes it's own class.
   # This class is similar to a dictionary, (e.g. loaded_data[key]).
    try:
        loaded_data = dict(np.load(file)) # lazy loading
    except FileNotFoundError:
        print(f"Error: File not found at path: {file}")
        return {}

    # Calculate the number of meshes from .npz file
    total_arrays = len(loaded_data)
    num_meshes = total_arrays // 3  # again, assume each mesh has 3 arrays: vertices, faces, colors
    mol_ids = [*range(1, num_meshes + 1)]
    
    # --- Multiprocessing ---
    # number of processes (use all CPUs if not specified)
    if num_processes is None:
        num_processes = mp.cpu_count()
    
    # partial function with fixed parameters
    # process_func = partial(import_one_mesh, loaded_data=loaded_data)
    
    # This function loops through every possible mol_id until reaching num_meshes,
    # (e.g. [1, 2, ..., 1501]). Also, it will return None if that mol_id is invalid
    meshes_dict = {}
    ctx = mp.get_context(context)
    with ctx.Pool(processes=num_processes, initializer=init_worker, initargs=(file,)) as pool:
        # Progress bar
        tqdm_iterator = tqdm(
            pool.imap(import_one_mesh, mol_ids),
            total=num_meshes,
            desc=f'Processing {num_meshes} molecules with {num_processes} logical cores',
            colour='#7BC8F6'
            )
        
        for mesh in tqdm_iterator:
            meshes_dict.update(mesh)
            
    return meshes_dict

In [ ]:
# Run the function to get the dictionary of meshes
all_reconstructed_meshes = import_meshes('npt-HK4_meshes.npz', num_processes=4, context='fork')

## Visualisation

In [8]:
# scene = trimesh.Scene(list(mol_meshes.values())[0:30])
# scene.show()